In [1]:
import os
import piexif
import pandas as pd
import numpy as np
from PIL import Image
from tqdm import tqdm
from scipy.stats import entropy
import cv2

In [2]:
valid_exts = (".jpg", ".jpeg", ".png", ".tif", ".tiff")

def extract_exif_info(image_path, img_format):
   if img_format not in ["JPEG", "TIFF"]:
       return {"Make": None, "Model": None, "Software": None, "DateTimeOriginal": None}
   try:
       exif_dict = piexif.load(image_path)
       exif_data = {}
       for ifd in exif_dict:
           for tag in exif_dict[ifd]:
               key = piexif.TAGS[ifd][tag]["name"]
               value = exif_dict[ifd][tag]
               if isinstance(value, bytes):
                   value = value.decode(errors="ignore")
               exif_data[key] = value
       return {
           "Make": exif_data.get("Make", None),
           "Model": exif_data.get("Model", None),
           "Software": exif_data.get("Software", None),
           "DateTimeOriginal": exif_data.get("DateTimeOriginal", None)
       }
   except:
       return {"Make": None, "Model": None, "Software": None, "DateTimeOriginal": None}

def extract_qtable(image_path, img_format):
   if img_format != "JPEG":
       return {"qtable_mean": None, "qtable_std": None, "qtable_max": None, "qtable_min": None}
   try:
       img = Image.open(image_path)
       qt = img.quantization
       if not qt:
           return {"qtable_mean": None, "qtable_std": None}
       q = list(qt.values())[0]
       q = np.array(q)
       return {
           "qtable_mean": np.mean(q),
           "qtable_std": np.std(q),
           "qtable_max": np.max(q),
           "qtable_min": np.min(q)
       }
   except:
       return {"qtable_mean": None, "qtable_std": None, "qtable_max": None, "qtable_min": None}

def extract_entropy(img_array):
   r, g, b = img_array[:, :, 0], img_array[:, :, 1], img_array[:, :, 2]
   return {
       "r_entropy": entropy(np.histogram(r, bins=256)[0] + 1e-7),
       "g_entropy": entropy(np.histogram(g, bins=256)[0] + 1e-7),
       "b_entropy": entropy(np.histogram(b, bins=256)[0] + 1e-7)
   }

def extract_blur_sharpness(img_array):
   gray = cv2.cvtColor(img_array, cv2.COLOR_RGB2GRAY)
   lap_var = cv2.Laplacian(gray, cv2.CV_64F).var()
   return {"blur_metric": lap_var}

def extract_hu_moments(img_array):
   gray = cv2.cvtColor(img_array, cv2.COLOR_RGB2GRAY)
   moments = cv2.moments(gray)
   hu_moments = cv2.HuMoments(moments).flatten()
   return {f"hu_{i+1}": float(val) for i, val in enumerate(hu_moments)}

def extract_dct_features(img_array):
    gray = cv2.cvtColor(img_array, cv2.COLOR_RGB2GRAY)
    h, w = gray.shape
    # Crop to even dimensions if necessary
    if h % 2 != 0:
        gray = gray[:-1, :]
    if w % 2 != 0:
        gray = gray[:, :-1]
    dct = cv2.dct(np.float32(gray) / 255.0)
    dct_abs = np.abs(dct)
    return {
        "dct_mean": np.mean(dct_abs),
        "dct_std": np.std(dct_abs),
        "dct_max": np.max(dct_abs),
        "dct_energy": np.sum(dct_abs ** 2)
    }

def extract_noise_features(img_array):
   gray = cv2.cvtColor(img_array, cv2.COLOR_RGB2GRAY)
   blurred = cv2.GaussianBlur(gray, (3, 3), 0)
   noise = gray.astype(np.float32) - blurred.astype(np.float32)
   return {
       "noise_mean": np.mean(noise),
       "noise_std": np.std(noise)
   }

def extract_edge_density(img_array):
   gray = cv2.cvtColor(img_array, cv2.COLOR_RGB2GRAY)
   edges = cv2.Canny(gray, 100, 200)
   return {
       "edge_density": np.sum(edges > 0) / edges.size
   }

def extract_image_stats(img):
   return {
       "width": img.width,
       "height": img.height
   }

def process_images(folder, label):
   data = []
   for fname in tqdm(os.listdir(folder), desc=f"Processing {folder}"):
       if not fname.lower().endswith(valid_exts):
           continue
       full_path = os.path.join(folder, fname)
       try:
           img = Image.open(full_path).convert("RGB")
           img_array = np.array(img)
           img_format = img.format or os.path.splitext(fname)[-1].replace(".", "").upper()

           row = {
               "filename": fname,
               "label": label,
               "format": img_format,
               "has_exif": img_format in ['JPEG', 'TIFF'],
               "has_qtable": img_format == 'JPEG'
           }

           row.update(extract_exif_info(full_path, img_format))
           row.update(extract_image_stats(img))
           row.update(extract_entropy(img_array))
           row.update(extract_blur_sharpness(img_array))
           row.update(extract_hu_moments(img_array))
           row.update(extract_dct_features(img_array))
           row.update(extract_noise_features(img_array))
           row.update(extract_edge_density(img_array))
           row.update(extract_qtable(full_path, img_format))

           data.append(row)
       except Exception as e:
           print(f"[ERROR] {fname}: {e}")
   return data

In [3]:
# Folder to label mapping
folder_label_map = {
    'human': 0,
    'edited': 1,
    'AI': 2
}

all_data = []

for folder, label in folder_label_map.items():
    folder_path = f'image_detection/{folder}'
    data = process_images(folder_path, label)
    all_data.extend(data)

# Convert to DataFrame and save
df = pd.DataFrame(all_data)
df.to_csv('image_detection/image_detection_metadata.csv', index=False)
print('Metadata extraction complete. Saved to image_detection_metadata.csv')

Processing image_detection/AI: 100%|██████████| 39975/39975 [15:10<00:00, 43.90it/s]


Metadata extraction complete. Saved to image_detection_metadata.csv


In [10]:
import os
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import torch
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [12]:
meta_features = [col for col in df.columns if col not in ['filename', 'label', 'format']]

# First split off the test set
trainval_df, test_df = train_test_split(df, test_size=0.2, stratify=df['label'], random_state=42)
# Then split train/val
train_df, val_df = train_test_split(trainval_df, test_size=0.25, stratify=trainval_df['label'], random_state=42)
# Now: train 60%, val 20%, test 20%

# Fit scaler on training metadata only
scaler = StandardScaler()
scaler.fit(train_df[meta_features])

# Transform all splits
train_df[meta_features] = scaler.transform(train_df[meta_features])
val_df[meta_features] = scaler.transform(val_df[meta_features])
test_df[meta_features] = scaler.transform(test_df[meta_features])

# Save splits
train_df.to_csv('image_detection/train_meta.csv', index=False)
val_df.to_csv('image_detection/val_meta.csv', index=False)
test_df.to_csv('image_detection/test_meta.csv', index=False)

import joblib
joblib.dump(scaler, 'image_detection/metadata_scaler.pkl')

/Users/wei/miniforge3/envs/nlp_env/lib/python3.10/site-packages/sklearn/utils/extmath.py:1144: RuntimeWarning: invalid value encountered in divide
  updated_mean = (last_sum + new_sum) / updated_sample_count
/Users/wei/miniforge3/envs/nlp_env/lib/python3.10/site-packages/sklearn/utils/extmath.py:1149: RuntimeWarning: invalid value encountered in divide
  T = new_sum / new_sample_count
/Users/wei/miniforge3/envs/nlp_env/lib/python3.10/site-packages/sklearn/utils/extmath.py:1169: RuntimeWarning: invalid value encountered in divide
  new_unnormalized_variance -= correction**2 / new_sample_count


['image_detection/metadata_scaler.pkl']

In [13]:
class HybridImageDataset(Dataset):
    def __init__(self, img_root, metadata_csv, transform=None, meta_features=None):
        self.img_root = img_root
        self.df = pd.read_csv(metadata_csv)
        self.transform = transform
        self.meta_features = meta_features or [col for col in self.df.columns if col not in ['filename', 'label', 'format']]
        
        # Fill missing values in metadata
        self.df[self.meta_features] = self.df[self.meta_features].fillna(self.df[self.meta_features].mean())

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        # Find which subfolder the image is in
        label = row['label']
        if label == 0:
            folder = 'human'
        elif label == 1:
            folder = 'edited'
        else:
            folder = 'AI'
        img_path = os.path.join(self.img_root, folder, row['filename'])
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        meta = torch.tensor(row[self.meta_features].values.astype(np.float32))
        label = torch.tensor(row['label']).long()
        return image, meta, label

In [14]:
import torch.nn as nn
import torchvision.models as models

class HybridNet(nn.Module):
    def __init__(self, num_metadata_features, num_classes=3):
        super().__init__()
        # Image branch: EfficientNet-B0
        self.cnn = models.efficientnet_b0(pretrained=True)
        self.cnn.classifier = nn.Identity()  # Remove final classification layer
        cnn_out_dim = 1280  # EfficientNet-B0 output features

        # Metadata branch
        self.meta_mlp = nn.Sequential(
            nn.Linear(num_metadata_features, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU()
        )

        # Fusion
        self.classifier = nn.Sequential(
            nn.Linear(cnn_out_dim + 32, 64),
            nn.ReLU(),
            nn.Linear(64, num_classes)
        )

    def forward(self, image, metadata):
        img_feat = self.cnn(image)
        meta_feat = self.meta_mlp(metadata)
        x = torch.cat([img_feat, meta_feat], dim=1)
        return self.classifier(x)

In [15]:
from torchvision import transforms

transform = transforms.Compose([
    transforms.Resize(256),                # Resize the shorter side to 256
    transforms.CenterCrop(224),            # Then crop the center 224x224
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [21]:
# Load meta features from your CSV
BATCH_SIZE = 32
df = pd.read_csv('image_detection/train_meta.csv')
meta_features = [col for col in df.columns if col not in ['filename', 'label', 'format']]

train_dataset = HybridImageDataset('image_detection', 'image_detection/train_meta.csv', transform, meta_features)
val_dataset = HybridImageDataset('image_detection', 'image_detection/val_meta.csv', transform, meta_features)
test_dataset = HybridImageDataset('image_detection', 'image_detection/test_meta.csv', transform, meta_features)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

In [17]:
if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Using MPS device")
else:
    device = torch.device("cpu")
    print("Using CPU device")

Using MPS device


In [22]:
from torch.optim import Adam
model = HybridNet(num_metadata_features=len(meta_features)).to(device)
optimizer = Adam(model.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss()

/Users/wei/miniforge3/envs/nlp_env/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/Users/wei/miniforge3/envs/nlp_env/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=EfficientNet_B0_Weights.IMAGENET1K_V1`. You can also use `weights=EfficientNet_B0_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [23]:
EPOCHS = 10

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0
    for images, metas, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]"):
        images, metas, labels = images.to(device), metas.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images, metas)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * images.size(0)
    train_loss /= len(train_loader.dataset)
    print(f"Epoch {epoch+1} Train Loss: {train_loss:.4f}")

    # Validation
    model.eval()
    val_loss = 0
    correct = 0
    with torch.no_grad():
        for images, metas, labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Val]"):
            images, metas, labels = images.to(device), metas.to(device), labels.to(device)
            outputs = model(images, metas)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)
            preds = outputs.argmax(1)
            correct += (preds == labels).sum().item()
    val_loss /= len(val_loader.dataset)
    val_acc = correct / len(val_loader.dataset)
    print(f"Epoch {epoch+1} Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

Epoch 1/10 [Train]: 100%|██████████| 1596/1596 [13:27<00:00,  1.98it/s]


Epoch 1 Train Loss: 0.1611


Epoch 1/10 [Val]: 100%|██████████| 532/532 [01:46<00:00,  5.01it/s]


Epoch 1 Val Loss: 0.0634 | Val Acc: 0.9781


Epoch 2/10 [Train]: 100%|██████████| 1596/1596 [13:11<00:00,  2.02it/s]


...

Epoch 9 Val Loss: 0.0555 | Val Acc: 0.9876


Epoch 10/10 [Train]: 100%|██████████| 1596/1596 [18:08<00:00,  1.47it/s]


Epoch 10 Train Loss: 0.0117


Epoch 10/10 [Val]: 100%|██████████| 532/532 [02:19<00:00,  3.81it/s]

Epoch 10 Val Loss: 0.0627 | Val Acc: 0.9855


In [24]:
model.eval()
test_loss = 0
correct = 0
all_preds = []
all_labels = []
with torch.no_grad():
    for images, metas, labels in tqdm(test_loader, desc="Testing"):
        images, metas, labels = images.to(device), metas.to(device), labels.to(device)
        outputs = model(images, metas)
        loss = criterion(outputs, labels)
        test_loss += loss.item() * images.size(0)
        preds = outputs.argmax(1)
        correct += (preds == labels).sum().item()
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
test_loss /= len(test_loader.dataset)
test_acc = correct / len(test_loader.dataset)
print(f"Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.4f}")

Testing: 100%|██████████| 532/532 [02:26<00:00,  3.63it/s]

Test Loss: 0.0627 | Test Acc: 0.9862


In [25]:
from sklearn.metrics import classification_report
print(classification_report(all_labels, all_preds, target_names=['human', 'edited', 'AI']))

              precision    recall  f1-score   support

       human       0.98      0.99      0.99      7995
      edited       0.97      0.92      0.94      1025
          AI       0.99      0.99      0.99      7995

    accuracy                           0.99     17015
   macro avg       0.98      0.97      0.97     17015
weighted avg       0.99      0.99      0.99     17015



In [26]:
torch.save(model.state_dict(), 'image_detection/hybrid_efficientnetb0_metadata.pth')